## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [1]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import mlflow
from mlflow.models import infer_signature
...

Ellipsis

Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [4]:
# Implement transformation logic as in session 2

df_validation = pd.read_csv("/Users/shelciamuianga/Desktop/MLOPS/mlops-and-system-design/Session 3/Churn_Modelling_train_test (1).csv")
df_validation_transformed = df_validation.copy() 



In [5]:
import joblib

PATH = "/Users/shelciamuianga/Desktop/MLOPS/mlops-and-system-design/Session 3/"

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

encoder.fit(df_validation_transformed[["Geography", "Gender"]])

joblib.dump(encoder, f'{PATH}one_hot_encoder.pkl')

['/Users/shelciamuianga/Desktop/MLOPS/mlops-and-system-design/Session 3/one_hot_encoder.pkl']

In [6]:
# Perform another experiment if you don't have the ones from session 2. Otherwise, this part can be skipped

# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [7]:
# import validation dataset to test inference
df_validation = pd.read_csv("/Users/shelciamuianga/Desktop/MLOPS/mlops-and-system-design/Session 3/Churn_Modelling_train_test (1).csv")

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [8]:
# transform data - if necessary

# load encoder
encoder = joblib.load(f'{PATH}one_hot_encoder.pkl')

# encode categorical columns
encoded_features = encoder.transform(df_validation[["Geography", "Gender"]])

# create dataframe for encoded columns
encoded_df = pd.DataFrame(
    encoded_features,
    columns=encoder.get_feature_names_out(["Geography", "Gender"])
)

# drop original categorical columns
df_validation_transformed = df_validation.drop(
    columns=["Geography", "Gender"]
)

# reset indexes
df_validation_transformed = df_validation_transformed.reset_index(drop=True)
encoded_df = encoded_df.reset_index(drop=True)

# concatenate encoded columns
df_validation_transformed = pd.concat(
    [df_validation_transformed, encoded_df],
    axis=1
)

##### Batch Inference

In [10]:
# define a function to implement batch inference with mlflow
def batch_inference(model_uri: str, input: pd.DataFrame):
    model = mlflow.pyfunc.load_model(model_uri)

    predictions = model.predict(input)

    return predictions

In [14]:
encoded_features = encoder.fit_transform(
    df_validation[["Geography", "Gender"]]
)

encoded_df = pd.DataFrame(
    encoded_features,
    columns=encoder.get_feature_names_out(["Geography", "Gender"])
)

df_validation_transformed = df_validation.drop(
    columns=["Geography", "Gender"]
)

df_validation_transformed = pd.concat(
    [df_validation_transformed.reset_index(drop=True),
     encoded_df.reset_index(drop=True)],
    axis=1
)

In [21]:
# define the model uri that should be used
model_uri = "mlflow-artifacts:/607637386112095231/a55be0e49475451ba7146098143f3dda/artifacts/bank_model"

# one-hot encode validation data
df_validation_transformed = pd.get_dummies(
    df_validation,
    columns=["Geography", "Gender"],
    drop_first=True
)

# remove target column
X_validation = df_validation_transformed.drop(columns=["Exited"])

# batch inference
batch_prediction_result = batch_inference(
    model_uri,
    X_validation
)

batch_prediction_result


2026/05/24 19:23:35 WARNING mlflow.models.utils: Found extra inputs in the model input that are not defined in the model signature: `['CustomerId', 'RowNumber', 'Surname']`. These inputs will be ignored.


array([0, 0, 1, ..., 1, 0, 0])

In [22]:
# check the confusion matrix
from sklearn.metrics import confusion_matrix

##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [23]:
import requests
import json

In [25]:
# import validation dataset to test inference - just one record
df_validation = pd.read_csv("bank-full_val.csv").head(1)

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [29]:
# transform data - if necessary
print(df_validation.columns.tolist())

['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


In [30]:
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

url = get_inference_endpoint()

In [31]:
# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(url: str, input: pd.DataFrame):

    response = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        json={
            "dataframe_split": input.to_dict(orient="split")
        }
    )

    return response

In [32]:
response_pandas = online_inference_pandas(
    url,
    X_validation.head(1)
)

response_pandas.content

b'{"predictions": [0]}'

In [33]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(url: str, input: dict):

    response = requests.post(
        url,
        headers={"Content-Type": "application/json"},
        json=input
    )

    return response

In [34]:
# define the json as required by MLFlow
input_json = {
    "dataframe_split": {
        "columns": X_validation.head(1).columns.tolist(),
        "data": X_validation.head(1).values.tolist()
    }
}

In [35]:
response_json = online_inference_json(
    url,
    input_json
)

response_json.content

b'{"predictions": [0]}'